# Week 10 · Notebook 1 — LangChain & LlamaIndex Building Blocks
**CSCI 250 — Introduction to Artificial Intelligence · Fall 2026**

Compose GenAI apps from reusable parts: **models → prompts → parsers → chains**, plus a quick **LlamaIndex** document index.

> Runs in Google Colab. Cells **degrade gracefully** if a package or API key is missing — you'll get a clear message instead of a crash. We use **Claude** and **Gemini** (never OpenAI).

## 0. Install frameworks

In [ ]:
!pip -q install langchain langchain-core langchain-community \
    langchain-anthropic langchain-google-genai \
    llama-index sentence-transformers

## 1. Load API keys safely
In Colab use the 🔑 *Secrets* panel; locally use environment variables. **Never commit keys.**

In [ ]:
import os
try:
    from google.colab import userdata
    for k in ('ANTHROPIC_API_KEY', 'GEMINI_API_KEY', 'GOOGLE_API_KEY'):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
except Exception:
    pass  # not on Colab — set keys in your shell
# langchain-google-genai reads GOOGLE_API_KEY
if os.environ.get('GEMINI_API_KEY') and not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = os.environ['GEMINI_API_KEY']
print('Anthropic key set:', bool(os.environ.get('ANTHROPIC_API_KEY')))
print('Google/Gemini key set:', bool(os.environ.get('GOOGLE_API_KEY')))

## 2. Prompt template + output parser
A **prompt template** is a reusable, parameterized prompt. An **output parser** turns the model's reply object into a plain string.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a concise tutor for a CS course.'),
    ('human', 'Explain {topic} to a {level} student in 2 sentences.'),
])
parser = StrOutputParser()
print(prompt.invoke({'topic': 'embeddings', 'level': 'beginner'}))

## 3. A chain with Claude (LCEL `|` pipe)
`chain = prompt | model | parser` — data flows left to right.

In [ ]:
try:
    from langchain_anthropic import ChatAnthropic
    claude = ChatAnthropic(model='claude-sonnet-4-6', max_tokens=300)
    chain = prompt | claude | parser
    print('CLAUDE:\n', chain.invoke({'topic': 'vector stores',
                                      'level': 'beginner'}))
except Exception as e:
    print('Claude chain skipped:', e)

## 4. The same chain with Gemini
Only the model changes — prompt and parser are reused. That is the payoff of the common interface.

In [ ]:
try:
    from langchain_google_genai import ChatGoogleGenerativeAI
    gemini = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
    chain = prompt | gemini | parser
    print('GEMINI:\n', chain.invoke({'topic': 'vector stores',
                                      'level': 'beginner'}))
except Exception as e:
    print('Gemini chain skipped:', e)

## 5. Document loaders & text splitters
Load text, then split it into overlapping **chunks** small enough to embed.

In [ ]:
sample = (
    'LangChain chains LLM steps with the pipe operator. '
    'LlamaIndex indexes documents for question answering. '
    'ChromaDB is an open-source vector store. '
    'Embeddings turn text into vectors that capture meaning. '
    'Retrieval-augmented generation grounds answers in your data.'
)
with open('notes.txt', 'w') as f:
    f.write(sample)

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = TextLoader('notes.txt').load()
splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
chunks = splitter.split_documents(docs)
print(len(chunks), 'chunks')
for c in chunks:
    print('-', c.page_content)

## 6. LlamaIndex — index a folder of docs
LlamaIndex hides loader → splitter → embed → store behind `VectorStoreIndex`. We point its embeddings at a **local Hugging Face** model (no OpenAI).

In [ ]:
import os
os.makedirs('docs', exist_ok=True)
with open('docs/notes.txt', 'w') as f:
    f.write(sample)

try:
    from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
    from llama_index.embeddings.huggingface import HuggingFaceEmbedding
    Settings.embed_model = HuggingFaceEmbedding(
        model_name='sentence-transformers/all-MiniLM-L6-v2')
    Settings.llm = None  # retrieval only — no LLM needed for this demo
    documents = SimpleDirectoryReader('docs').load_data()
    index = VectorStoreIndex.from_documents(documents)
    retriever = index.as_retriever(similarity_top_k=2)
    hits = retriever.retrieve('What stores vectors?')
    for h in hits:
        print(round(h.score, 3), '|', h.node.get_content())
except Exception as e:
    print('LlamaIndex demo skipped (install llama-index-embeddings-huggingface):', e)

## 7. Recap
- **Models** are swappable behind one interface (Claude ↔ Gemini).
- **Prompt templates + parsers + chains** remove glue code.
- **Loaders + splitters** prepare your documents.
- **LlamaIndex** packages indexing into a few lines.

Next notebook: embeddings and a **ChromaDB** vector store you can query.